<a href="https://colab.research.google.com/github/Williamzamo/Trabajos-algebra-lineal/blob/main/Descomposicion_por_rango.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto computacional: Descomposición por rango

**Curso:** Álgebra Lineal II — Programa de Matemáticas Aplicadas y Computacionales, UMNG
**Profesor:** Juan Camilo Torres Chaves

Este notebook desarrolla la teoría detrás de la **descomposición por rango** de una matriz y presenta la solución computacional a los dos proyectos propuestos:

1. Dada una matriz $A$, hallar matrices $C$ y $R$ tales que $A = CR$, con $C$ y $R$ de rango completo.
2. Dada una matriz $A$ de rango $r$, expresarla como una suma de $r$ matrices de rango $1$.


## 1. Teoría: ¿qué es la descomposición por rango?

Toda matriz $A \in M_{m\times n}(\mathbb{R})$ de rango $r$ admite una factorización

$$A = CR,$$

donde $C \in M_{m\times r}(\mathbb{R})$ y $R \in M_{r\times n}(\mathbb{R})$ son ambas de **rango completo** $r$
(es decir, $C$ tiene $r$ columnas linealmente independientes y $R$ tiene $r$ filas linealmente independientes).

## 2. Metodología para calcular $C$ y $R$

Una forma estándar de obtener la descomposición por rango es a través de la forma escalonada reducida
por filas (RREF) de $A$:

1. Calculamos $E = \mathrm{RREF}(A)$.
2. Identificamos las **columnas pivote** de $E$ (las que contienen el primer $1$ de cada fila no nula).
3. $C$ se forma tomando, de la matriz original $A$ (no de $E$), las columnas que corresponden a esos
   índices pivote.
4. $R$ se forma tomando las primeras $r$ filas de $E$ (las filas no nulas de la forma escalonada), donde
   $r$ es el número de columnas pivote (el rango de $A$).

La razón por la que esto funciona: las columnas pivote de $A$ son linealmente independientes y generan el
mismo espacio columna que $A$; y toda columna de $A$ se puede escribir como combinación lineal de las
columnas pivote usando exactamente los coeficientes que aparecen en las columnas de $E$. Esa es
justamente la información codificada en $R$.

A continuación implementamos esto en código, siguiendo el guion de la matriz `A` de ejemplo.


In [ ]:
import numpy as np

# Matriz de ejemplo sobre la que ilustraremos la descomposición.
# Tiene tamaño 3x4. Sus filas satisfacen: fila3 = fila1 + fila2 (dependencia lineal),
# así que se espera que el rango sea menor que 3.
# Para probar con otras matrices, basta con modificar A.
A = np.array([[1, 2, 0, 3],
              [2, 4, 1, 8],
              [3, 6, 1, 11]], dtype=float)

A


### Paso 1: Eliminación gaussiana para obtener la RREF

La función `rref` implementa el algoritmo de Gauss-Jordan:

- Recorre columna por columna buscando un pivote (una entrada distinta de cero) en la fila actual.
- Si la entrada en la posición del pivote es cero, intercambia filas hasta encontrar una fila con
  entrada no nula en esa columna.
- Si toda la columna (de la fila actual hacia abajo) es cero, avanza a la siguiente columna sin avanzar
  de fila.
- Normaliza la fila pivote dividiendo por el valor del pivote (para que quede un $1$).
- Elimina la entrada en esa columna en **todas** las demás filas (no solo abajo, sino también arriba),
  para dejar la matriz en forma escalonada **reducida**.


In [ ]:
def rref(matrix):
    # Copiamos la matriz para no modificar la original y trabajamos en punto flotante.
    a = matrix.astype(float).copy()
    filas, columnas = a.shape
    pivote = 0  # índice de la columna donde buscamos el próximo pivote

    for r in range(filas):
        if pivote >= columnas:
            break  # ya no quedan columnas por revisar

        i = r
        # Buscamos una fila (desde la actual hacia abajo) con entrada no nula en la columna 'pivote'
        while a[i, pivote] == 0:
            i += 1
            if i == filas:
                # No hay pivote en esta columna: pasamos a la siguiente columna
                # sin avanzar de fila.
                i = r
                pivote += 1
                if columnas == pivote:
                    return a  # ya recorrimos todas las columnas
        # Intercambiamos filas para traer el pivote a la posición r
        a[[i, r]] = a[[r, i]]
        # Normalizamos la fila pivote para que la entrada pivote quede en 1
        a[r] = a[r] / a[r, pivote]
        # Eliminamos la columna pivote en todas las demás filas (reducción completa)
        for i in range(filas):
            if i != r:
                a[i] -= a[i, pivote] * a[r]
        pivote += 1
    return a

print("RREF(A):")
print(np.round(rref(A), 4))


### Paso 2: identificar las columnas pivote

Una vez tenemos $E = \mathrm{RREF}(A)$, las columnas pivote son las que contienen el primer elemento no
nulo de cada fila no nula de $E$. Estas columnas indican, en la matriz original $A$, cuáles columnas son
linealmente independientes.


In [ ]:
def columnas_pivote(a):
    matriz_escalonada = rref(a)
    tol = 1e-9  # tolerancia numérica para considerar una entrada como "cero"
    columnas_piv = []
    for fila in matriz_escalonada:
        indices_no_cero = np.where(np.abs(fila) > tol)[0]
        if len(indices_no_cero) > 0:
            # el primer índice no nulo de la fila es la columna pivote de esa fila
            columnas_piv.append(int(indices_no_cero[0]))
    return columnas_piv

print("Índices de columnas pivote:", columnas_pivote(A))


### Paso 3: construir $C$ y $R$

- $C$: tomamos de la matriz **original** $A$ las columnas cuyos índices son columnas pivote.
- $R$: tomamos de $E = \mathrm{RREF}(A)$ las primeras $r$ filas (las filas no nulas), donde $r$ es la
  cantidad de columnas pivote encontradas.


In [ ]:
def obtener_C(a):
    cols_piv = columnas_pivote(a)
    # Apilamos como columnas las columnas pivote de la matriz original
    return np.column_stack([a[:, j] for j in cols_piv])

def obtener_R(a):
    r = len(columnas_pivote(a))
    E = rref(a)
    # Las primeras r filas de la RREF son exactamente las filas no nulas
    return E[:r, :]

def descomposicion_CR(a):
    return obtener_C(a), obtener_R(a)

C, R = descomposicion_CR(A)
print("C =")
print(np.round(C, 4))
print("\nR =")
print(np.round(R, 4))
print(f"\nrank(A) = {C.shape[1]}")


## 3. Segundo proyecto: descomposición en matrices de rango 1

Toda matriz de rango $r$ se puede escribir como una suma de $r$ matrices de rango $1$:

$$A = A_1 + A_2 + \cdots + A_r.$$

La idea es directa a partir de $A = CR$: si $C = [c_1 \; c_2 \; \cdots \; c_r]$ tiene como columnas los
vectores $c_k \in \mathbb{R}^m$, y $R$ tiene como filas los vectores fila $\rho_k \in \mathbb{R}^n$
($k=1,\dots,r$), entonces el producto matricial $CR$ es exactamente

$$CR = \sum_{k=1}^{r} c_k \rho_k,$$

donde cada término $c_k \rho_k$ (producto exterior de un vector columna por un vector fila) es una matriz
de tamaño $m\times n$ y **rango exactamente 1** (siempre que $c_k \neq 0$ y $\rho_k \neq 0$).

Esto es simplemente la definición de multiplicación de matrices "por columnas de $C$ contra filas de $R$",
en lugar de la definición habitual "fila por columna".


In [ ]:
def descomposicion_rango1(a):
    C, R = descomposicion_CR(a)
    r = C.shape[1]
    matrices = []
    for k in range(r):
        col = C[:, k].reshape(-1, 1)   # k-ésima columna de C, como vector columna
        fila = R[k, :].reshape(1, -1)  # k-ésima fila de R, como vector fila
        matrices.append(col @ fila)    # producto exterior: matriz de rango 1
    return matrices

matrices_rango1 = descomposicion_rango1(A)
for k, Ak in enumerate(matrices_rango1, start=1):
    print(f"A{k} (rango 1):")
    print(np.round(Ak, 4))
    print()


## 4. Programa completo (todo junto)

La siguiente celda reúne todas las piezas anteriores en un único script ejecutable, tal como se entregaría
como solución final del proyecto. Está fuertemente comentado para poder explicar cada paso durante la
sustentación.


In [1]:
import numpy as np

# ---- Datos de entrada ----
# Para probar con otras matrices, basta con modificar A.
A = np.array([[1, 2, 0, 3],
              [2, 4, 1, 8],
              [3, 6, 1, 11]], dtype=float)


def rref(matrix):
    '''Calcula la forma escalonada reducida por filas (Gauss-Jordan).'''
    a = matrix.astype(float).copy()
    filas, columnas = a.shape
    pivote = 0
    for r in range(filas):
        if pivote >= columnas:
            break
        i = r
        while a[i, pivote] == 0:
            i += 1
            if i == filas:
                i = r
                pivote += 1
                if columnas == pivote:
                    return a
        a[[i, r]] = a[[r, i]]
        a[r] = a[r] / a[r, pivote]
        for i in range(filas):
            if i != r:
                a[i] -= a[i, pivote] * a[r]
        pivote += 1
    return a


def columnas_pivote(a):
    '''Índices de las columnas pivote de la RREF de a.'''
    matriz_escalonada = rref(a)
    tol = 1e-9
    columnas_piv = []
    for fila in matriz_escalonada:
        indices_no_cero = np.where(np.abs(fila) > tol)[0]
        if len(indices_no_cero) > 0:
            columnas_piv.append(int(indices_no_cero[0]))
    return columnas_piv


def obtener_C(a):
    '''C: columnas pivote de la matriz ORIGINAL a.'''
    cols_piv = columnas_pivote(a)
    return np.column_stack([a[:, j] for j in cols_piv])


def obtener_R(a):
    '''R: primeras r filas (no nulas) de la RREF de a.'''
    r = len(columnas_pivote(a))
    E = rref(a)
    return E[:r, :]


def descomposicion_CR(a):
    '''Proyecto computacional 1: retorna (C, R) tales que a = C @ R.'''
    return obtener_C(a), obtener_R(a)


def descomposicion_rango1(a):
    '''Proyecto computacional 2: retorna lista [A1, ..., Ar] con a = A1 + ... + Ar,
    donde cada Ak tiene rango 1.'''
    C, R = descomposicion_CR(a)
    r = C.shape[1]
    matrices = []
    for k in range(r):
        col = C[:, k].reshape(-1, 1)
        fila = R[k, :].reshape(1, -1)
        matrices.append(col @ fila)
    return matrices


def mostrar_matriz(nombre, M):
    print(f"\n{nombre}")
    print(np.round(M, 4))


def main():
    mostrar_matriz("Matriz A", A)
    mostrar_matriz("RREF(A)", rref(A))

    C, R = descomposicion_CR(A)
    mostrar_matriz("C", C)
    mostrar_matriz("R", R)
    print(f"\nrank(A) = {C.shape[1]}")

    matrices = descomposicion_rango1(A)
    for k, Ak in enumerate(matrices, start=1):
        mostrar_matriz(f"A{k} rank 1", Ak)

    suma = sum(matrices)
    print(f"\nsuma de las {len(matrices)} matrices rank 1 (verificacion)\n{suma}")


main()



Matriz A
[[ 1.  2.  0.  3.]
 [ 2.  4.  1.  8.]
 [ 3.  6.  1. 11.]]

RREF(A)
[[1. 2. 0. 3.]
 [0. 0. 1. 2.]
 [0. 0. 0. 0.]]

C
[[1. 0.]
 [2. 1.]
 [3. 1.]]

R
[[1. 2. 0. 3.]
 [0. 0. 1. 2.]]

rank(A) = 2

A1 rank 1
[[1. 2. 0. 3.]
 [2. 4. 0. 6.]
 [3. 6. 0. 9.]]

A2 rank 1
[[0. 0. 0. 0.]
 [0. 0. 1. 2.]
 [0. 0. 1. 2.]]

suma de las 2 matrices rank 1 (verificacion)
[[ 1.  2.  0.  3.]
 [ 2.  4.  1.  8.]
 [ 3.  6.  1. 11.]]
